# Tentativo 18 · 2. TF-IDF e Mistral 24B: regressori e classificatori, cinque condizioni

Due lettori del testo:
- **TF-IDF**: parole e coppie di parole (100.000), vocabolario imparato sui soli testi di training;
- **Mistral-Small 24B congelato**: il vettore di 5.120 numeri di ogni articolo (strato 27, calcolato nel tentativo 16).

Sopra, due modelli lineari:
- **regressore**: ridge sul punteggio 0–100;
- **classificatore**: logistica sui 5 livelli, trasformata in punteggio (la media dei punteggi di ogni livello nel training,
  pesata con le probabilità): serve a vedere se prevedere il numero serve davvero.

Cinque **condizioni dei dati** (cambiano solo l'addestramento; il test sono sempre tutti gli articoli delle testate di test):

| | condizione |
|---|---|
| A | tutti gli articoli (chi scrive di più pesa di più) |
| B | tutti gli articoli, **ogni testata pesa uguale** (peso 1 / articoli della testata) |
| C | al più **10 articoli** per testata, presi a caso |
| D | al più **30 articoli** per testata |
| E | **un vettore per testata** (la media dei vettori dei suoi articoli) |

La penalità (alpha della ridge, C della logistica) si sceglie con una validazione interna sulle testate di training
(3 fold sul punteggio, errore medio assoluto per testata). Tre split, cinque fold: ogni testata è prevista da un modello
che non l'ha mai vista. Circa un'ora sul server (solo CPU).

In [1]:
T16_DIR = "../16_DataPreparation"
N_JOBS = 32
MODELLI = "TF-IDF,Mistral 24B"
MISTRAL = "Results/embedding/Mistral-Small-24B-Base-2501_strato27"     # nel tentativo 16

In [2]:
# Parameters
T16_DIR = "/home/fabio.pinelli/John/Tentativo16"


In [3]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

T16 = Path(T16_DIR).expanduser(); sys.path.insert(0, str(T16))
import funzioni as F16                      # il tentativo 16: dataset e cartella dei dati
import punteggio as P

warnings.filterwarnings("ignore")
plt.rcParams.update(F16.STILE_GRAFICI)
BLU, ARANCIO, GRIGIO, CHIARO, VERDE, ROSSO = "#2a78d6", "#eb6834", "#52514e", "#b5b3ab", "#1baf7a", "#b02a2a"
DATI18 = F16.DATI / "tentativi" / "18_PunteggioEsatto"; DATI18.mkdir(parents=True, exist_ok=True)
OUT = Path("Results"); OUT.mkdir(exist_ok=True)
art = pd.read_csv(F16.DATASET / "articoli.csv.gz")
t = pd.read_csv(F16.DATASET / "testate.csv", index_col="domain")
print(f"{len(art):,} articoli di {len(t)} testate")

25,524 articoli di 226 testate


In [4]:
from joblib import Parallel, delayed
split = pd.read_csv(DATI18 / "split.csv", index_col=0).reindex(t.index)
X = np.load(T16 / MISTRAL / "embedding.npy")
assert np.array_equal(np.load(T16 / MISTRAL / "embedding_article_id.npy"), art.article_id.to_numpy())
art["score"] = t.score.reindex(art.domain).to_numpy()
LAVORI = OUT / "lineari" / "lavori"; LAVORI.mkdir(parents=True, exist_ok=True)
lavori = [(m, s, k, c, tipo) for m in MODELLI.split(",") for s in split.columns for k in range(1, 6) for c in P.CONDIZIONI
          for tipo in ["regressore", "classificatore"]]
nome = lambda m, s, k, c, tipo: LAVORI / f"{m.replace(' ', '')}_{s}_f{k}_{c}_{tipo}.csv.gz"
mancano = [l for l in lavori if not nome(*l).exists()]
print(f"{len(lavori)} lavori, {len(mancano)} da fare")


def esegui(m, s, k, c, tipo):
    out, testate, info = P.un_lavoro(m, X if m != "TF-IDF" else None, art, split[s], k, c, tipo, int(s[-1]))
    out.assign(modello=m, split=s, condizione=c, tipo=tipo).to_csv(nome(m, s, k, c, tipo), index=False)
    if testate is not None:
        testate.assign(modello=m, split=s, condizione=c, tipo=tipo).to_csv(str(nome(m, s, k, c, tipo)).replace(".csv.gz", "_testate.csv.gz"), index=False)
    pd.DataFrame([info]).to_csv(str(nome(m, s, k, c, tipo)).replace(".csv.gz", "_info.csv"), index=False)
    return True

# prima i lavori di TF-IDF (vocabolario per fold, più lenti), poi Mistral
Parallel(n_jobs=N_JOBS, verbose=5)(delayed(esegui)(*l) for l in sorted(mancano, key=lambda l: l[0] != "TF-IDF"))
prev = pd.concat([pd.read_csv(nome(*l)) for l in lavori], ignore_index=True)
prev.to_csv(OUT / "lineari" / "previsioni_articoli.csv.gz", index=False)
tes = pd.concat([pd.read_csv(str(nome(*l)).replace(".csv.gz", "_testate.csv.gz")) for l in lavori if l[3] == "E"], ignore_index=True)
tes.to_csv(OUT / "lineari" / "previsioni_testate_E.csv.gz", index=False)
scelte = pd.concat([pd.read_csv(str(nome(*l)).replace(".csv.gz", "_info.csv")) for l in lavori], ignore_index=True)
scelte.to_csv(OUT / "lineari" / "penalita_scelte.csv", index=False)
print(f"{len(prev):,} previsioni per articolo")

300 lavori, 300 da fare


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.


[Parallel(n_jobs=32)]: Done   8 tasks      | elapsed:  1.4min


[Parallel(n_jobs=32)]: Done  98 tasks      | elapsed: 10.5min


[Parallel(n_jobs=32)]: Done 224 tasks      | elapsed: 22.2min


[Parallel(n_jobs=32)]: Done 298 out of 300 | elapsed: 32.8min remaining:   13.2s


[Parallel(n_jobs=32)]: Done 300 out of 300 | elapsed: 33.1min finished


1,531,440 previsioni per articolo


## La penalità scelta
Quante volte ogni valore della griglia è stato scelto, per modello, tipo e condizione.

In [5]:
scelte.groupby(["modello", "tipo", "condizione"]).penalità.value_counts().unstack(fill_value=0)

penalità                               0.001      0.010      0.030      \
modello     tipo           condizione                                    
Mistral 24B classificatore A                   1          7          0   
                           B                   4          8          0   
                           C                   0          2          0   
                           D                   0          4          0   
                           E                   3         12          0   
            regressore     A                   0          0          0   
                           B                   0          0          0   
                           C                   0          0          0   
                           D                   0          0          0   
                           E                   0          0          0   
TF-IDF      classificatore A                   0          0          0   
                           B                   0          0          0   
                           C                   0          0          0   
                           D                   0          0          0   
                           E                   0          0          0   
            regressore     A                   0          0         15   
                           B                   0          0         10   
                           C                   0          0         15   
                           D                   0          0         15   
                           E                   0          0         15   

penalità                               0.100      0.300      1.000      \
modello     tipo           condizione                                    
Mistral 24B classificatore A                   6          0          1   
                           B                   3          0          0   
                           C                   8          0          5   
                           D                   8          0          3   
                           E                   0          0          0   
            regressore     A                   0          0          0   
                           B                   0          0          0   
                           C                   0          0          0   
                           D                   0          0          0   
                           E                   0          0          0   
TF-IDF      classificatore A                   0          0          0   
                           B                   0          0          0   
                           C                   0          0          0   
                           D                   0          0          0   
                           E                   0          0          0   
            regressore     A                   0          0          0   
                           B                   4          1          0   
                           C                   0          0          0   
                           D                   0          0          0   
                           E                   0          0          0   

penalità                               100.000    1000.000   10000.000  
modello     tipo           condizione                                   
Mistral 24B classificatore A                   0          0          0  
                           B                   0          0          0  
                           C                   0          0          0  
                           D                   0          0          0  
                           E                   0          0          0  
            regressore     A                   0          8          7  
                           B                   0          4         11  
                           C                   0         15          0  
            